In [1]:
from pyspark.sql import SparkSession
spark=(
    SparkSession.builder
    .appName("RDD Transformation")
    .master("local[*]")
    .getOrCreate()
)
sc=spark.sparkContext
print(sc.uiWebUrl)

sc

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/17 21:01:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://macbookair.lan:4040


<SparkContext master=local[*] appName=RDD Transformation>

# Narrow Transformation 

In [ ]:
# map() - This is an RDD transformation that applies a function to each element of an RDD and returns exactly one output element for each input element.

# map() -> 1 input -> 1 output 

#new_rdd=rdd.map(function)

numbers=[10,20,30,40,50]
rdd=sc.parallelize(numbers)
result=rdd.map(lambda x:x*2)
result.collect()


In [ ]:
def multiple_by_10(x):
    return x*10

In [ ]:
result1=rdd.map(multiple_by_10)
result1.collect()

In [ ]:
# Important 

rdd=sc.parallelize([
    "Vijay",
    "Vamshi",
    "Darshan",
    "Ashad"
])

In [ ]:
result=rdd.map(lambda name:(name,len(name)))

In [ ]:
result.collect()

In [ ]:
row_data=[
    "T001,C101,5000,India",
    "T002,C102,6000,Usa",
    "T003,C103,59000,uk",
    "T004,C104,5400,Usa",
    "T005,C105,50400,India"

]

In [ ]:
rdd_row_data=sc.parallelize(row_data)
result_row_data=rdd_row_data.map(lambda x:x.split(","))
result_row_data.collect()

In [ ]:
def parse_transformation(record):
    txn_id,customer_id,amount,country=record.split(",")
    return(
        txn_id,
        customer_id,
        float(amount),
        country.upper()
    )

In [ ]:
transection_rdd=rdd_row_data.map(parse_transformation)


In [ ]:
transection_rdd.collect()

### Map() Usecases 

- Parsing records 
- Type Conversion 
- Extacting Values 
- Adding calculated fields 
- Converting one struture into another 
- Creating key-value RDD
- Applying Bsuiness Logic 



In [ ]:
transections=[
    ('T001', 'C101', 5000.0, 'INDIA'),
    ('T002', 'C102', 6000.0, 'USA'),
    ('T003', 'C103', 59000.0, 'UK'),
    ('T004', 'C104', 5400.0, 'USA'),
    ('T005', 'C105', 50400.0, 'INDIA')]
rdd=sc.parallelize(transections)

In [ ]:
customer_amount=rdd.map(lambda x:(x[1],x[2]))
customer_amount.collect()

In [ ]:
rdd=sc.parallelize([1,2,3])
result=rdd.map(lambda x:[x,x*10])
result.collect()

In [ ]:
# Map() and Task Execution 

rdd=sc.parallelize(range(1_000_000),4)
result=rdd.map(lambda x:x*2)
result.collect()

In [ ]:
# Multiple Narrow Transformation 

rdd=sc.parallelize(range(100),4)
rdd2=rdd.map(lambda x:x*2)
rdd3=rdd2.map(lambda x:x*10)
rdd4=rdd3.filter(lambda x:x>50)
rdd4.collect()

## Questions map() Transformation 

Q1 — Employee Salary Hike

employees = [
    ("E101", "An", "Data Engineer", 80000),
    ("E102", "Rahul", "Developer", 60000),
    ("E103", "Priya", "Data Engineer", 90000),
    ("E104", "Amit", "Tester", 50000)
]

rdd = sc.parallelize(employees)


- Using only map(), apply these hikes:

Data Engineer → 15%
Developer     → 10%
Tester        → 5%

(employee_id, name, role, old_salary, new_salary)


Q2 — Transaction Risk Classification

transactions = [
    ("T001", "C101", 5000, "India"),
    ("T002", "C102", 18000, "USA"),
    ("T003", "C103", 75000, "India"),
    ("T004", "C104", 25000, "UK"),
    ("T005", "C105", 120000, "USA")
]

rdd = sc.parallelize(transactions)

Rules: 

amount < 10,000              → LOW
10,000 <= amount < 50,000    → MEDIUM
amount >= 50,000             → HIGH

Expected Structure:

(transaction_id, customer_id, amount, country, risk)

Q3 — Parse Raw CSV Records 

raw_data = [
    "E101,Anuj,Data Engineer,85000,India",
    "E102,Rahul,Developer,65000,USA",
    "E103,Priya,Manager,120000,India"
]

rdd = sc.parallelize(raw_data)


- Using map(), convert each string into a tuple and convert salary to int.
- Also uppercase the country.


Q4 — E-commerce Order Calculation

orders = [
    ("O101", "Laptop", 2, 50000),
    ("O102", "Mouse", 5, 1000),
    ("O103", "Keyboard", 3, 2000),
    ("O104", "Monitor", 2, 15000)
]

rdd = sc.parallelize(orders)

Fields:

(order_id, product, quantity, unit_price)

Calculate:

total_amount = quantity × unit_price
GST          = total_amount × 18%
final_amount = total_amount + GST

Expected Structure:

(order_id, product, total_amount, gst, final_amount)

-- Example :

("O101", "Laptop", 100000, 18000, 118000)
("O102", "Mouse",    5000,   900,   5900)


Q5 — Create a Pair RDD for Future Aggregation

sales = [
    ("S001", "C101", "Laptop", 50000),
    ("S002", "C102", "Mobile", 30000),
    ("S003", "C101", "Keyboard", 5000),
    ("S004", "C103", "Monitor", 20000),
    ("S005", "C102", "Mouse", 2000)
]

rdd = sc.parallelize(sales)

Transform this into a Pair RDD where:

Key   = Customer ID
Value = (Product, Amount)


Expected:

("C101", ("Laptop", 50000))
("C102", ("Mobile", 30000))
("C101", ("Keyboard", 5000))
("C103", ("Monitor", 20000))
("C102", ("Mouse", 2000))

Q6 — Data Quality Flagging

customers = [
    ("C101", "Anuj", 32, "anuj@gmail.com"),
    ("C102", "", 28, "rahul@gmail.com"),
    ("C103", "Priya", -5, "priya@gmail.com"),
    ("C104", "Amit", 45, ""),
    ("C105", "Neha", 25, "neha@gmail.com")
]

rdd = sc.parallelize(customers)


Rules:

name is empty  → INVALID_NAME

age <= 0       → INVALID_AGE

email is empty → INVALID_EMAIL

otherwise      → VALID


Expected Struccure:

(customer_id, name, age, email, status)

Expected:

("C101", "Anuj",  32, "anuj@gmail.com",  "VALID")
("C102", "",      28, "rahul@gmail.com", "INVALID_NAME")
("C103", "Priya", -5, "priya@gmail.com", "INVALID_AGE")
("C104", "Amit",  45, "",                 "INVALID_EMAIL")
("C105", "Neha",  25, "neha@gmail.com",  "VALID")


Q7 — Multiple Business Rules

transactions = [
    ("T001", "C101", 5000, "India", "UPI"),
    ("T002", "C102", 60000, "India", "CARD"),
    ("T003", "C103", 120000, "USA", "CARD"),
    ("T004", "C104", 8000, "UK", "CASH"),
    ("T005", "C105", 90000, "India", "UPI")
]

rdd = sc.parallelize(transactions)

Create :

(transaction_id,
 customer_id,
 amount,
 country,
 payment_method,
 risk_score,
 risk_category)

 Risk Score:

 amount >= 100000        → +3
amount >= 50000         → +2
otherwise               → +1

country != "India"      → +2

payment_method == CARD  → +1

Risk category:

score >= 5 → HIGH
score >= 3 → MEDIUM
otherwise  → LOW


Example:

T003
Amount 120000 → +3
USA           → +2
CARD          → +1
                 --
Score            6

Risk = HIGH


Expected T003: ("T003", "C103", 120000, "USA", "CARD", 6, "HIGH")






In [ ]:
# ## Questions map() Transformation 

# Q1 — Employee Salary Hike

# employees = [
#     ("E101", "An", "Data Engineer", 80000),
#     ("E102", "Rahul", "Developer", 60000),
#     ("E103", "Priya", "Data Engineer", 90000),
#     ("E104", "Amit", "Tester", 50000)
# ]

# rdd = sc.parallelize(employees)


# - Using only map(), apply these hikes:

# Data Engineer → 15%
# Developer     → 10%
# Tester        → 5%

# (employee_id, name, role, old_salary, new_salary)


# Q2 — Transaction Risk Classification

# transactions = [
#     ("T001", "C101", 5000, "India"),
#     ("T002", "C102", 18000, "USA"),
#     ("T003", "C103", 75000, "India"),
#     ("T004", "C104", 25000, "UK"),
#     ("T005", "C105", 120000, "USA")
# ]

# rdd = sc.parallelize(transactions)

# Rules: 

# amount < 10,000              → LOW
# 10,000 <= amount < 50,000    → MEDIUM
# amount >= 50,000             → HIGH

# Expected Structure:

# (transaction_id, customer_id, amount, country, risk)

# Q3 — Parse Raw CSV Records 

# raw_data = [
#     "E101,Anuj,Data Engineer,85000,India",
#     "E102,Rahul,Developer,65000,USA",
#     "E103,Priya,Manager,120000,India"
# ]

# rdd = sc.parallelize(raw_data)


# - Using map(), convert each string into a tuple and convert salary to int.
# - Also uppercase the country.


# Q4 — E-commerce Order Calculation

# orders = [
#     ("O101", "Laptop", 2, 50000),
#     ("O102", "Mouse", 5, 1000),
#     ("O103", "Keyboard", 3, 2000),
#     ("O104", "Monitor", 2, 15000)
# ]

# rdd = sc.parallelize(orders)

# Fields:

# (order_id, product, quantity, unit_price)

# Calculate:

# total_amount = quantity × unit_price
# GST          = total_amount × 18%
# final_amount = total_amount + GST

# Expected Structure:

# (order_id, product, total_amount, gst, final_amount)

# -- Example :

# ("O101", "Laptop", 100000, 18000, 118000)
# ("O102", "Mouse",    5000,   900,   5900)


# Q5 — Create a Pair RDD for Future Aggregation

# sales = [
#     ("S001", "C101", "Laptop", 50000),
#     ("S002", "C102", "Mobile", 30000),
#     ("S003", "C101", "Keyboard", 5000),
#     ("S004", "C103", "Monitor", 20000),
#     ("S005", "C102", "Mouse", 2000)
# ]

# rdd = sc.parallelize(sales)

# Transform this into a Pair RDD where:

# Key   = Customer ID
# Value = (Product, Amount)


# Expected:

# ("C101", ("Laptop", 50000))
# ("C102", ("Mobile", 30000))
# ("C101", ("Keyboard", 5000))
# ("C103", ("Monitor", 20000))
# ("C102", ("Mouse", 2000))

# Q6 — Data Quality Flagging

# customers = [
#     ("C101", "Anuj", 32, "anuj@gmail.com"),
#     ("C102", "", 28, "rahul@gmail.com"),
#     ("C103", "Priya", -5, "priya@gmail.com"),
#     ("C104", "Amit", 45, ""),
#     ("C105", "Neha", 25, "neha@gmail.com")
# ]

# rdd = sc.parallelize(customers)


# Rules:

# name is empty  → INVALID_NAME

# age <= 0       → INVALID_AGE

# email is empty → INVALID_EMAIL

# otherwise      → VALID


# Expected Struccure:

# (customer_id, name, age, email, status)

# Expected:

# ("C101", "Anuj",  32, "anuj@gmail.com",  "VALID")
# ("C102", "",      28, "rahul@gmail.com", "INVALID_NAME")
# ("C103", "Priya", -5, "priya@gmail.com", "INVALID_AGE")
# ("C104", "Amit",  45, "",                 "INVALID_EMAIL")
# ("C105", "Neha",  25, "neha@gmail.com",  "VALID")


# Q7 — Multiple Business Rules

# transactions = [
#     ("T001", "C101", 5000, "India", "UPI"),
#     ("T002", "C102", 60000, "India", "CARD"),
#     ("T003", "C103", 120000, "USA", "CARD"),
#     ("T004", "C104", 8000, "UK", "CASH"),
#     ("T005", "C105", 90000, "India", "UPI")
# ]

# rdd = sc.parallelize(transactions)

# Create :

# (transaction_id,
#  customer_id,
#  amount,
#  country,
#  payment_method,
#  risk_score,
#  risk_category)

#  Risk Score:

#  amount >= 100000        → +3
# amount >= 50000         → +2
# otherwise               → +1

# country != "India"      → +2

# payment_method == CARD  → +1

# Risk category:

# score >= 5 → HIGH
# score >= 3 → MEDIUM
# otherwise  → LOW


# Example:

# T003
# Amount 120000 → +3
# USA           → +2
# CARD          → +1
#                  --
# Score            6

# Risk = HIGH


# Expected T003: ("T003", "C103", 120000, "USA", "CARD", 6, "HIGH")




# # 

In [ ]:
# flatmap() : Applies a function to every element in RDD but one input element can produce zero,one or multiple output element.

# map() - I input -> 1 Output 

# flatmap() - 1 Input -> 0,1, or many outputs 

# flatmap() -> MAP() -> Transform each element   Flat-> Flatten the results 



In [ ]:
rdd=sc.parallelize([
    "Apache Spark",
    "Data Engineering",
    "Big Data"
])

In [ ]:
result=rdd.map(lambda x:x.split(" "))
result.collect()

In [ ]:
result=rdd.flatMap(lambda x:x.split(" "))
result.collect()

In [ ]:
rdd=sc.parallelize([
    "Spark",
    "",
    "Python"
])

In [ ]:
result=rdd.flatMap(lambda x:[x] if x != "" else [])

In [ ]:
result.collect()

In [ ]:
# Does flatMap() changes the Number of partition - No 

rdd=sc.parallelize([
    "Apache Spark",
    "Data Engineering",
    "Big Data"
],3)

rdd.getNumPartitions()




In [ ]:
result=rdd.flatMap(lambda x:x.split(" "))
result.getNumPartitions()

In [ ]:
result.collect()

In [ ]:
logs=[
    "ERROR database connection Failed",
    "INFO application started",
    "ERROR payment service timeout"
]

rdd=sc.parallelize(logs)

In [ ]:
words=rdd.flatMap(lambda line:line.split())

In [ ]:
words.collect()

In [ ]:
orders=[
    ("0101",["Laptop","Mouse"]),
    ("0102",["Keyboard"]),
    ("0103",["Monitor","Mouse","Keyboard"])
]

rdd=sc.parallelize(orders)

In [ ]:
result=rdd.flatMap(
    lambda x:[(x[0],product) for product in x[1]]
)

result.collect()

In [ ]:
customer=[
    ("C101",[5000,3000,7000]),
    ("C102",[10000,20000]),
    ("C103",[]),

]
rdd=sc.parallelize(customer)

In [ ]:
result=rdd.flatMap(
    lambda x:[(x[0],amount) for amount in x[1]]
)
result.collect()

In [ ]:
x=("C101",[])
output=[]
for amount in x[1]:
    output.append((x[0],amount))

In [ ]:
output

In [ ]:
# filter() : Returns only those RDD elements that staisfy a condition 

# new_rdd=rdd.filter(function)

# new_rdd=rdd.filter(lambda x: condition)

rdd=sc.parallelize([10,15,20,25,30])
result=rdd.filter(lambda x:x>20)
result.collect()



In [ ]:
rdd=sc.parallelize([1,2,3,4,5,6,7,7,8,8,8,78,6,6,56,5,4,4,3,3,3])

In [ ]:
result=rdd.filter(lambda x:x%2==0)
result.collect()

In [ ]:
logs=sc.parallelize([
    "INFO Application started",
    "ERROR Database connection failed",
    "WARN memory useage high",
    "ERROR payment service timeout",
    "INFO Application completed"
])



In [ ]:
errors=logs.filter(
    lambda line : "ERROR" in line
)

errors.collect()

# ── WIDE TRANSFORMATIONS
SET / DATASET
distinct()
intersection()
subtract()

In [ ]:
# distinct() : Removes duplicate elemnts from an RDD

# distinct(numPartitions) : The desired number of partition for the result \

-   rdd.distinct(numPartitions=10)

In [2]:
rdd=sc.parallelize([
    10,20,10,30,20,40
])
result=rdd.distinct()
result.collect()

[40, 10, 20, 30]

In [3]:
events=sc.parallelize([
    ("U101","Product_1"),
    ("U102","Product_2"),
    ("U101","Product_1"),
    ("U103","Product_3"),
    ("U102","Product_2"),
    ("U102","Product_2"),
    ("U102","Product_2"),
    ("U101","Product_1"),
])

# Find unique (user,product) interatction 

unique_events=events.distinct()
unique_events.collect()


[('U102', 'Product_2'), ('U103', 'Product_3'), ('U101', 'Product_1')]

1 TB -> 500 Partitions 

rdd.distinct()

Spark may need significant:

- Network I/O 
- Disk I/O
- Serialization 
- Shuffle Read/write 
- memory 

### Intersection - Finds the elements thta are present in both RDDs

RDD_1 and RDD_2 --> Common element 

rdd1.intersection(rdd2)





subtract() : - Returns elements that are present in the RDD but not in the second RDD

rdd.subtract(rdd2)


